In [1]:
import pennylane as qml
from pennylane import numpy as np
from itertools import combinations

n = 4
wires = list(range(n))
gamma = 0.7

dev = qml.device("default.qubit", wires=n)

# Method 1: Pauli-Z expansion
def U_C_pauli(gamma):
    for i, j in combinations(wires, 2):
        qml.PauliRot(-gamma / 4, "ZZ", wires=[i, j])

    qml.PauliRot(-gamma / 4, "ZZZZ", wires=wires)


# Method 2: None/all projectors
def U_C_projector(gamma):
    # Phase on |1111>
    qml.ctrl(
        qml.PhaseShift,
        control=wires[:-1],
        control_values=[1, 1, 1]
    )(gamma, wires=wires[-1])

    # Phase on |0000>
    # X on target converts zero-control to one-control
    qml.PauliX(wires=wires[-1])
    qml.ctrl(
        qml.PhaseShift,
        control=wires[:-1],
        control_values=[0, 0, 0]
    )(gamma, wires=wires[-1])
    qml.PauliX(wires=wires[-1])

In [2]:
U_pauli = qml.matrix(U_C_pauli, wire_order=wires)(gamma)
U_proj  = qml.matrix(U_C_projector, wire_order=wires)(gamma)

# U_pauli = exp(-i gamma * 7/8) * U_proj
global_phase = np.exp(-1j * 7 * gamma / 8)

print("Maximum matrix difference:")
print(np.max(np.abs(U_pauli - global_phase * U_proj)))

Maximum matrix difference:
0.5189914418890909


In [3]:
@qml.qnode(dev)
def pauli_circuit(gamma):
    U_C_pauli(gamma)
    return qml.state()

@qml.qnode(dev)
def projector_circuit(gamma):
    U_C_projector(gamma)
    return qml.state()

print("PAULI CIRCUIT")
print(qml.draw(pauli_circuit)(gamma))

print("\nPROJECTOR CIRCUIT")
print(qml.draw(projector_circuit)(gamma))

PAULI CIRCUIT
0: ─╭RZZ(-0.17)─╭RZZ(-0.17)─╭RZZ(-0.17)─────────────────────────────────────╭RZZZZ(-0.17)─┤  State
1: ─╰RZZ(-0.17)─│───────────│───────────╭RZZ(-0.17)─╭RZZ(-0.17)─────────────├RZZZZ(-0.17)─┤  State
2: ─────────────╰RZZ(-0.17)─│───────────╰RZZ(-0.17)─│───────────╭RZZ(-0.17)─├RZZZZ(-0.17)─┤  State
3: ─────────────────────────╰RZZ(-0.17)─────────────╰RZZ(-0.17)─╰RZZ(-0.17)─╰RZZZZ(-0.17)─┤  State

PROJECTOR CIRCUIT
0: ─╭●───────────╭○───────────┤  State
1: ─├●───────────├○───────────┤  State
2: ─├●───────────├○───────────┤  State
3: ─╰Rϕ(0.70)──X─╰Rϕ(0.70)──X─┤  State


In [4]:
gate_set = {
    "CNOT", "Hadamard", "PauliX",
    "RX", "RY", "RZ", "PhaseShift",
    "T", "Adjoint(T)", "S", "Adjoint(S)",
    "GlobalPhase"
}

pauli_decomp = qml.transforms.decompose(
    pauli_circuit,
    gate_set=gate_set
)

projector_decomp = qml.transforms.decompose(
    projector_circuit,
    gate_set=gate_set
)

In [5]:
print("PAULI DECOMPOSITION")
print(qml.draw(pauli_decomp)(gamma))

print("\nPROJECTOR DECOMPOSITION")
print(qml.draw(projector_decomp)(gamma))

PAULI DECOMPOSITION
0: ─╭X──RZ(-0.17)─╭X─╭X──RZ(-0.17)─╭X─╭X──RZ(-0.17)─╭X───────────────────────────────────── ···
1: ─╰●────────────╰●─│─────────────│──│─────────────│──╭X──RZ(-0.17)─╭X─╭X──RZ(-0.17)─╭X─── ···
2: ──────────────────╰●────────────╰●─│─────────────│──╰●────────────╰●─│─────────────│──╭X ···
3: ───────────────────────────────────╰●────────────╰●──────────────────╰●────────────╰●─╰● ···

0: ··· ─────────────────────╭X──RZ(-0.17)─╭X───────┤  State
1: ··· ──────────────────╭X─╰●────────────╰●─╭X────┤  State
2: ··· ──RZ(-0.17)─╭X─╭X─╰●──────────────────╰●─╭X─┤  State
3: ··· ────────────╰●─╰●────────────────────────╰●─┤  State

PROJECTOR DECOMPOSITION
0: ───────────╭●───────────╭●────╭●──T─────────╭●───────────────────╭GlobalPhase(-0.00)─── ···
1: ────╭●─────│─────╭●─────│───T─╰X──T†────────╰X───────────────────├GlobalPhase(-0.00)─── ···
2: ────│──────│─────│──────│────────────────────────────────────────├GlobalPhase(-0.00)─╭● ···
3: ──H─╰X──T†─╰X──T─╰X──T†─╰X──T──H──RZ(12.48

In [6]:
for name, circuit in [
    ("Pauli", pauli_decomp),
    ("Projector", projector_decomp)
]:
    resources = qml.specs(circuit)(gamma)["resources"]

    print(f"\n{name}")
    print("CNOT count:", resources.gate_types.get("CNOT", 0))
    print("Total gates:", resources.num_gates)
    print("Circuit depth:", resources.depth)
    print("Gate types:", resources.gate_types)


Pauli
CNOT count: 18
Total gates: 25
Circuit depth: 22
Gate types: {'CNOT': 18, 'RZ': 7}

Projector
CNOT count: 40
Total gates: 154
Circuit depth: 127
Gate types: {'Hadamard': 8, 'CNOT': 40, 'Adjoint(T)': 12, 'T': 16, 'RZ': 32, 'RY': 16, 'GlobalPhase': 16, 'PhaseShift': 6, 'PauliX': 8}


In [8]:
U1 = qml.matrix(pauli_decomp, wire_order=wires)(gamma)
U2 = qml.matrix(projector_decomp, wire_order=wires)(gamma)

error = np.max(
    np.abs(U1 - np.exp(-1j * gamma / 8) * U2)
)

print("Maximum difference:", error)

Maximum difference: 4.833954429062804e-15
